In [1]:
%pip install pandas
%pip innstall matplotlob
%pip install seaborn

You should consider upgrading via the '/Users/ehlamkin/Desktop/nir_2026_v2/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
ERROR: unknown command "innstall" - maybe you meant "install"
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/ehlamkin/Desktop/nir_2026_v2/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
from parse_apparatus import parse_docx_diffs
from split_manuscripts import split_manuscripts
import pandas as pd
from dataclasses import asdict


## Получаю правильные диффы по словам из Глава 43_091test.docx

In [3]:
docx_path = 'data/source/Глава 43_091test.docx' 
output_dir = 'data/clean'

true_values = parse_docx_diffs(docx_path, "GB")
true_values[0:3]

[{'op_type': 'replace',
  'base_token': 'възвеселе',
  'other_token': 'възвеселїе',
  'char_ops': []},
 {'op_type': 'equal',
  'base_token': 'высоты',
  'other_token': 'высоты',
  'char_ops': []},
 {'op_type': 'replace',
  'base_token': 'ѹтвержⷣенїа',
  'other_token': 'ѹтвръжденїе',
  'char_ops': []}]

In [4]:
## Ctl+F-ом нашел 25 упоминаний про P81 в тексте, а тут - 19
i = 0
for item in true_values:
    if item['op_type'] != 'equal':
        i+=1

i

29

Все отличия типа replace меду M10 и P81:

In [5]:
for item in true_values:
    if item['op_type'] == 'replace':
        print(item)

{'op_type': 'replace', 'base_token': 'възвеселе', 'other_token': 'възвеселїе', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'ѹтвержⷣенїа', 'other_token': 'ѹтвръжденїе', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'дѣла', 'other_token': 'дѣло', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'дыма', 'other_token': 'домы', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'огнь', 'other_token': 'ѡгньно', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'сътворшїи', 'other_token': 'съвтори', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'свѣтило', 'other_token': 'свѣтоло', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'полнъ', 'other_token': 'плѣнъ', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'другѫ', 'other_token': 'дѫгѫ', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'сътворишаго', 'other_token': 'створшаго', 'char_ops': []}
{'op_type': 'replace', 'base_token': 'окрочеⷩ\ue003еⷨ', 'other_token': 'ѿкроченїем', 'char_ops': []}
{'o

## Создаю отдельные документики с отдельным текстом в каждом

In [6]:
docx_path = 'data/source/Глава 43_t.docx' 
output_dir = 'data/clean'

split_manuscripts(docx_path, output_dir)

Saved M10 to data/clean/M10.docx (34 fragments)
Saved MN to data/clean/MN.docx (34 fragments)
Saved P81 to data/clean/P81.docx (34 fragments)
Saved P78 to data/clean/P78.docx (34 fragments)
Saved GB to data/clean/GB.docx (34 fragments)
Saved IV13 to data/clean/IV13.docx (34 fragments)
Saved IV605 to data/clean/IV605.docx (34 fragments)
Saved U3 to data/clean/U3.docx (34 fragments)
Saved GB2 to data/clean/GB2.docx (34 fragments)
Saved Tr730 to data/clean/Tr730.docx (34 fragments)
Saved TB to data/clean/TB.docx (34 fragments)
Saved МИД181 to data/clean/МИД181.docx (34 fragments)


## Для созданных текстов прогоняю алгоритм Левенштейна

In [7]:
import sys
import os
import docx
import re
# Добавляем корневую папку проекта в путь, чтобы импортировать algo
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from algo.comparison import compare_tokens
from algo.structures import Weights

def read_docx(path):
    doc = docx.Document(path)
    return " ".join([p.text for p in doc.paragraphs])

def tokenize(text):
    return re.findall(r'\S+', text)

# 1. Читаем тексты
text_m10 = read_docx('data/clean/M10.docx')
text_p81 = read_docx('data/clean/P81.docx')

# 2. Токенизируем
tokens_m10 = tokenize(text_m10)
tokens_p81 = tokenize(text_p81)

# 3. Сравниваем (можно выбрать "myers", "levenshtein", "levenshtein_nested")
test_values = compare_tokens(tokens_m10, tokens_p81, Weights(1, 1, 1), algorithm="levenshtein_nested")

Все отличия типа replace меду M10 и P81:

In [8]:
for item in test_values:
    if item.op_type == 'replace':
        print(item)

WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=0, other_index=0, base_token='възвеселе', other_token='възвеселїе')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=8, other_index=8, base_token='сл҃нце', other_token='сл҃нца')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=29, other_index=29, base_token='дыма', other_token='дымы')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=106, other_index=106, base_token='другѫ', other_token='дѹгꙋ')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=116, other_index=116, base_token='н҃бо', other_token='нб҃ѹ')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=196, other_index=195, base_token='земли', other_token='землю')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=199, other_index=198, base_token='съмерзнетсѧ', other_token='съмръзшисѧ')
WordOp(op_type=<WordOpType.replace: 'replace'>, base_index=234, other_index=234, base_token='всему', other_token='емѹ')
WordOp(op_type=<Wor

## Немного проверок

In [9]:
t = 79
print(true_values[t])
print(test_values[t])


{'op_type': 'equal', 'base_token': 'высотѣ', 'other_token': 'высотѣ', 'char_ops': []}
WordOp(op_type=<WordOpType.equal: 'equal'>, base_index=79, other_index=78, base_token='на', other_token='на')


In [10]:
print(len(test_values))
print(len(true_values))

362
352


In [11]:
r = []
for item in test_values:
    r += [item.op_type]
set(r)

{<WordOpType.delete: 'delete'>,
 <WordOpType.equal: 'equal'>,
 <WordOpType.insert: 'insert'>,
 <WordOpType.replace: 'replace'>}

In [12]:
r = []
for item in true_values:
    r += [item['op_type']]
set(r)

{'equal', 'replace'}

## Метрики точности
* Выбираю только replace, потому что другие виды изменений по-нормальному не распарсить

## Общая функция для расчета метрик для пары текстов

In [13]:
def get_metrics(text1, text2, true_values_path='data/source/Глава 43_091test.docx', mode='prod'):
    
    def read_docx(path):
        doc = docx.Document(path)
        return " ".join([p.text for p in doc.paragraphs])

    def tokenize(text):
        return re.findall(r'\S+', text)

    # 1. Читаем тексты
    text_m10 = read_docx(f'data/clean/{text1}.docx')
    text_p81 = read_docx(f'data/clean/{text2}.docx')

    # 2. Токенизируем
    tokens_m10 = tokenize(text_m10)
    tokens_p81 = tokenize(text_p81)

    # 3. Сравниваем (можно выбрать "myers", "levenshtein", "levenshtein_nested")
    test_values = compare_tokens(tokens_m10, tokens_p81, Weights(1, 1, 1), algorithm="levenshtein_nested")
    true_values = parse_docx_diffs(true_values_path, text2)

    # 1. Преобразуем test_values (список объектов) в список словарей
    # Используем asdict, так как WordOp — это dataclass
    test_dicts = [asdict(op) for op in test_values]

    # 2. Создаем DataFrame'ы
    df_true = pd.DataFrame(true_values)
    df_test = pd.DataFrame(test_dicts)

    #3. Выбираю только replace
    df_true_replace = df_true[df_true['op_type'] == 'replace'].reset_index(drop=True)
    df_test_replace = df_test[df_test['op_type'] == 'replace'].reset_index(drop=True)

    # Для recall: true_df как база (левый датафрейм)
    result_recall = pd.merge(
        df_true_replace,  # левый датафрейм (все записи из true)
        df_test_replace,  # правый датафрейм
        on=['base_token', 'other_token'], 
        how='left',
        suffixes=('_true', '_test')  # добавляем суффиксы для различения колонок
    )[['op_type_true', 'op_type_test', 'base_token', 'other_token']]

    total_true = len(result_recall)  # Всего записей в истинном датафрейме
    found_in_test = result_recall['op_type_test'].notna().sum()  # Сколько нашлось в тестовом
    recall = found_in_test / total_true if total_true > 0 else 0

    # Для precision: test_df как база (левый датафрейм)
    result_precision = pd.merge(
        df_test_replace,   # левый датафрейм (все записи из test)
        df_true_replace,   # правый датафрейм
        on=['base_token', 'other_token'], 
        how='left',
        suffixes=('_test', '_true')  # добавляем суффиксы для различения колонок
    )[['op_type_test', 'op_type_true', 'base_token', 'other_token']]

    total_test = len(result_precision)  # Всего записей в тестовом датафрейме
    found_in_true = result_precision['op_type_true'].notna().sum()  # Сколько подтвердилось в истинном
    precision = found_in_true / total_test if total_test > 0 else 0

    print(f"Recall: {recall:.2%} ({found_in_test}/{total_true})")
    print(f"Precision: {precision:.2%} ({found_in_true}/{total_test})")

    if mode == 'prod':
        return recall, precision
    elif mode == 'debug':
        return result_recall, result_precision
    


In [14]:
res = get_metrics("M10", "GB", mode='debug')

Recall: 82.76% (24/29)
Precision: 80.00% (24/30)


In [16]:
res[0].tail()

,op_type_true,op_type_test,base_token,other_token
24,replace,WordOpType.replace,кїтово,китове
25,replace,WordOpType.replace,бл҃гоюхаⷩа,блг҃оѫханїе
26,replace,WordOpType.replace,їмат,имамъ
27,replace,WordOpType.replace,възносїте,възнесѣте
28,replace,NaN,їмате,имате бѡ


In [17]:
get_metrics("M10", "GB", mode='prod')

Recall: 82.76% (24/29)
Precision: 80.00% (24/30)


(np.float64(0.8275862068965517), np.float64(0.8))

In [18]:
sigles = ['МИД181', 'GB', 'GB2', 'IV13', 'IV605', 'P78', 'P81', 'TB', 'Tr730', 'U3']
origin = ['M10' for _ in range(len(sigles))]
d = {
    'origin': [],
    'compare': [],
    'recall': [],
    'precision': []
}
for sigl in sigles:
    d['origin'] += ['M10']
    d['compare'] += [sigl]
    comp = get_metrics("M10", sigl, mode='prod')
    d['recall'] += [comp[0]]
    d['precision'] += [comp[1]]
metrics = pd.DataFrame(d)
print('recall avg: ', metrics['recall'].mean())
print('precision avg: ', metrics['precision'].mean())
metrics

Recall: 66.13% (41/62)
Precision: 48.24% (41/85)
Recall: 82.76% (24/29)
Precision: 80.00% (24/30)
Recall: 92.31% (24/26)
Precision: 80.00% (24/30)
Recall: 87.50% (14/16)
Precision: 63.64% (14/22)
Recall: 86.67% (13/15)
Precision: 65.00% (13/20)
Recall: 86.67% (13/15)
Precision: 81.25% (13/16)
Recall: 73.68% (14/19)
Precision: 73.68% (14/19)
Recall: 80.00% (44/55)
Precision: 70.97% (44/62)
Recall: 63.16% (12/19)
Precision: 54.55% (12/22)
Recall: 76.00% (19/25)
Precision: 65.52% (19/29)
recall avg:  0.7948707838519032
precision avg:  0.6828363061405751


,origin,compare,recall,precision
0,M10,МИД181,0.661290,0.482353
1,M10,GB,0.827586,0.800000
2,M10,GB2,0.923077,0.800000
3,M10,IV13,0.875000,0.636364
4,M10,IV605,0.866667,0.650000
5,M10,P78,0.866667,0.812500
6,M10,P81,0.736842,0.736842
7,M10,TB,0.800000,0.709677
8,M10,Tr730,0.631579,0.545455
9,M10,U3,0.760000,0.655172


In [20]:
metrics

,origin,compare,recall,precision
0,M10,МИД181,0.661290,0.482353
1,M10,GB,0.827586,0.800000
2,M10,GB2,0.923077,0.800000
3,M10,IV13,0.875000,0.636364
4,M10,IV605,0.866667,0.650000
5,M10,P78,0.866667,0.812500
6,M10,P81,0.736842,0.736842
7,M10,TB,0.800000,0.709677
8,M10,Tr730,0.631579,0.545455
9,M10,U3,0.760000,0.655172


In [21]:
metrics.rename(columns={'origin': 'Основной текст', 'compare': 'Сравниваемый текст'})

,Основной текст,Сравниваемый текст,recall,precision
0,M10,МИД181,0.661290,0.482353
1,M10,GB,0.827586,0.800000
2,M10,GB2,0.923077,0.800000
3,M10,IV13,0.875000,0.636364
4,M10,IV605,0.866667,0.650000
5,M10,P78,0.866667,0.812500
6,M10,P81,0.736842,0.736842
7,M10,TB,0.800000,0.709677
8,M10,Tr730,0.631579,0.545455
9,M10,U3,0.760000,0.655172
